In [ ]:
from google.colab import drive
import os


drive.mount('/content/drive')


DRIVE_SAVE_DIR = "/content/drive/MyDrive/Vegetable_Models"

if not os.path.exists(DRIVE_SAVE_DIR):
    os.makedirs(DRIVE_SAVE_DIR)
    print(f"Created directory: {DRIVE_SAVE_DIR}")
else:
    print(f"Directory exists: {DRIVE_SAVE_DIR}")

In [ ]:
import os
from google.colab import files

!pip install -q kaggle

print("Please upload your kaggle.json file:")
uploaded = files.upload()

if not os.path.exists('/root/.kaggle'):
    !mkdir -p ~/.kaggle

!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d misrakahmed/vegetable-image-dataset
!unzip -q vegetable-image-dataset.zip

base_path = "/content/Vegetable Images"
train_path = os.path.join(base_path, "train")
val_path   = os.path.join(base_path, "validation")
test_path  = os.path.join(base_path, "test")

print("Data paths set. Training directory:", train_path)

Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/misrakahmed/vegetable-image-dataset
License(s): CC-BY-SA-4.0
 99% 529M/534M [00:04<00:00, 123MB/s] 
100% 534M/534M [00:04<00:00, 118MB/s]
Data paths set. Training directory: /content/Vegetable Images/train


In [ ]:
train_path = "/content/Vegetable Images/train"
val_path = "/content/Vegetable Images/validation"
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess_input
import os

EPOCHS = 10
NUM_CLASSES = 15
IMG_SIZE = (224,224)
BATCH = 32

#pre procc for custom and mobile
print("🔹 Setting up Standard Data Generators (Rescale 1/255)...")
train_std_gen = ImageDataGenerator(
    rescale=1./255, rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1, horizontal_flip=True
)
val_std_gen = ImageDataGenerator(rescale=1./255)

train_ds = train_std_gen.flow_from_directory(
    train_path, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical'
)
val_ds = val_std_gen.flow_from_directory(
    val_path, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical'
)

#pre procc for effici
print("\n🔹 Setting up EfficientNet Data Generators (preprocess_input)...")
train_eff_gen = ImageDataGenerator(
    preprocessing_function=eff_preprocess_input,
    rotation_range=15, width_shift_range=0.1,
    height_shift_range=0.1, zoom_range=0.1, horizontal_flip=True
)
val_eff_gen = ImageDataGenerator(preprocessing_function=eff_preprocess_input)

train_eff_ds = train_eff_gen.flow_from_directory(
    train_path, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical'
)
val_eff_ds = val_eff_gen.flow_from_directory(
    val_path, target_size=IMG_SIZE, batch_size=BATCH, class_mode='categorical'
)


def get_model_smart(model_name, create_model_func, train_data, val_data):

    model_path = f"{DRIVE_SAVE_DIR}/{model_name}.h5"

    if os.path.exists(model_path):
        print(f"\n🔄 Found existing {model_name} in Drive. Loading...")
        return load_model(model_path)
    else:
        print(f"\n🆕 Model {model_name} not found in Drive. Starting training...")
        model = create_model_func()
        model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])


        model.fit(train_data, validation_data=val_data, epochs=EPOCHS)


        model.save(model_path)
        print(f"✅ Saved {model_name} to {model_path}")
        return model




def create_cnn():
    return models.Sequential([
        layers.Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
        layers.MaxPooling2D(),
        layers.Conv2D(64, (3,3), activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, (3,3), activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])


model_cnn = get_model_smart("custom_cnn", create_cnn, train_ds, val_ds)



def create_efficientnet():
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        input_shape=(224,224,3),
        weights="imagenet"
    )
    base.trainable = False

    return models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])


model_eff = get_model_smart("efficientnet", create_efficientnet, train_eff_ds, val_eff_ds)



def create_mobilenet():
    base = tf.keras.applications.MobileNetV2(
        include_top=False,
        input_shape=(224,224,3),
        weights="imagenet"
    )
    base.trainable = False

    return models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation="softmax")
    ])


model_mob = get_model_smart("mobilenetv2", create_mobilenet, train_ds, val_ds)

print("\n🎉 All 3 models (CNN, EfficientNet, MobileNet) are processed and saved in Drive!")

🔹 Setting up Standard Data Generators (Rescale 1/255)...
Found 15000 images belonging to 15 classes.
Found 3000 images belonging to 15 classes.

🔹 Setting up EfficientNet Data Generators (preprocess_input)...
Found 15000 images belonging to 15 classes.
Found 3000 images belonging to 15 classes.

🔄 Found existing custom_cnn in Drive. Loading...



🔄 Found existing efficientnet in Drive. Loading...



🔄 Found existing mobilenetv2 in Drive. Loading...



🎉 All 3 models (CNN, EfficientNet, MobileNet) are processed and saved in Drive!


In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image


MODEL_PATH = "/content/drive/MyDrive/Vegetable_Models/efficientnet.h5"

@st.cache_resource
def load_my_model():
    return tf.keras.models.load_model(MODEL_PATH, compile=False)

model = load_my_model()


CLASS_NAMES = ['Bean', 'Bitter_Gourd', 'Bottle_Gourd', 'Brinjal', 'Broccoli', 'Cabbage', 'Capsicum', 'Carrot', 'Cauliflower', 'Cucumber', 'Papaya', 'Potato', 'Pumpkin', 'Radish', 'Tomato']

st.title("Vegetable Classifier 🥦")

uploaded_file = st.file_uploader("Choose an image...", type="jpg")

if uploaded_file is not None:
    image = Image.open(uploaded_file)
    st.image(image, caption='Uploaded Image.', use_column_width=True)

    img = image.resize((224, 224))
    img_array = np.array(img)


    from tensorflow.keras.applications.efficientnet import preprocess_input

    img_array = preprocess_input(img_array)


    img_array = np.expand_dims(img_array, axis=0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    predicted_index = np.argmax(predictions[0])
    predicted_class = CLASS_NAMES[predicted_index]

    st.write(f"### Prediction: {predicted_class}")


Writing app.py


In [ ]:
!pip install -q pyngrok
!pip install streamlit
from pyngrok import ngrok
import threading
import time
import subprocess
import os
from google.colab import userdata

ngrok.kill()

# Try to get the token from Colab secrets first
NGROK_TOKEN = userdata.get('NGROK_TOKEN')

# If not found in Colab secrets, try environment variables
if NGROK_TOKEN is None:
    NGROK_TOKEN = os.environ.get("NGROK_TOKEN")

if NGROK_TOKEN:
    print("ngrok token loaded successfully.")
    ngrok.set_auth_token(NGROK_TOKEN)
else:
    raise ValueError(
        "NGROK_TOKEN not found in Colab secrets or environment variables.\n"
        "Set it up via the key icon (Secrets) in the left sidebar, name it 'NGROK_TOKEN'."
    )

public_url = ngrok.connect(8501)
print("Your public URL:", public_url)

def run_streamlit():
    subprocess.call(["streamlit", "run", "app.py", "--server.port=8501"])

thread = threading.Thread(target=run_streamlit)
thread.start()

time.sleep(5)
print("Streamlit is running...")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
